# Install Requirements

In [21]:
!pip install selenium webdriver-manager dateparser tqdm requests ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 3.4 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 2.9 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: ok


# Location Discovery

In [4]:
import csv
import logging
import os
import random
import sys
import threading
import time
import re
from datetime import datetime
from queue import Queue
from typing import List, Tuple

import requests
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service

# ------------------------------
# KONFIGURASI
# ------------------------------
CONFIG = {
    "target_provinces": ["DKI Jakarta", "Banten", "Jawa Barat", "Jawa Timur"],
    "nation": "Indonesia",
    "max_retries": 3,
    "wait_seconds": 5,
    "headless": True,
    "respectful_delay_min": 0.8,
    "respectful_delay_max": 2.0,
    "user_agents": [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edge/120.0.0.0",
    ],
    "robots_root": "https://www.olx.co.id",
}

# URL provinsi langsung (struktur baru OLX)
PROVINCE_URLS = {
    "Jawa Timur": "https://www.olx.co.id/jawa-timur_g2000011/dijual-rumah-apartemen_c5158",
    "DKI Jakarta": "https://www.olx.co.id/jakarta-dki_g2000007/dijual-rumah-apartemen_c5158",
    "Banten": "https://www.olx.co.id/banten_g2000004/dijual-rumah-apartemen_c5158",
    "Jawa Barat": "https://www.olx.co.id/jawa-barat_g2000009/dijual-rumah-apartemen_c5158",
}

# ------------------------------
# LOGGING
# ------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s:%(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("olx-location-discovery")

# ------------------------------
# GLOBAL QUEUE
# ------------------------------
WRITE_QUEUE: "Queue[dict]" = Queue()

# ------------------------------
# UTILITAS
# ------------------------------
def respectful_pause():
    time.sleep(random.uniform(CONFIG["respectful_delay_min"], CONFIG["respectful_delay_max"]))

def pick_user_agent():
    return random.choice(CONFIG["user_agents"])

def make_driver(user_agent: str = None):
    # 🟡 Pastikan msedgedriver.exe ada di folder kerja saat ini
    driver_path = os.path.join(os.getcwd(), "msedgedriver.exe")
    service = Service(driver_path)

    options = Options()
    if CONFIG["headless"]:
        options.add_argument("--headless=new")
    options.add_argument("--inprivate")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    if user_agent:
        options.add_argument(f"--user-agent={user_agent}")

    # ✅ Inisialisasi driver dengan service & options
    driver = webdriver.Edge(service=service, options=options)
    driver.set_page_load_timeout(CONFIG["wait_seconds"] * 2)
    return driver

def safe_get(driver, url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            driver.get(url)
            return True
        except (WebDriverException, TimeoutException) as e:
            wait = (2 ** (attempt - 1)) + random.random()
            logger.warning(f"get({url}) attempt {attempt} failed: {e} — backoff {wait:.1f}s")
            time.sleep(wait)
            continue
    return False

# ------------------------------
# SCRAPING ANCHORS
# ------------------------------

LISTING_URL_PATTERN = re.compile(
    r"^https://(www\.)?olx\.co\.id/[a-z0-9-]+_g\d+/dijual-rumah-apartemen_c5158/?$"
)

def find_location_anchors(driver) -> List[Tuple[str, str]]:
    anchors = []
    try:
        body_as = driver.find_elements(By.CSS_SELECTOR, "main a, div a, section a")
        for a in body_as:
            txt = (a.text or "").strip()
            href = a.get_attribute("href")
            if txt and href:
                href = href.strip()
                # ✅ Filter hanya link ke halaman properti (bukan promo/news)
                if LISTING_URL_PATTERN.match(href):
                    anchors.append((txt, href))
    except Exception:
        pass

    # Hilangkan duplikat
    seen = set()
    uniq = []
    for t, h in anchors:
        key = (t.lower(), h)
        if key not in seen:
            seen.add(key)
            uniq.append((t, h))
    return uniq

# ------------------------------
# EXTRACT LINKS
# ------------------------------
def extract_province_links(driver, desired_provinces: List[str]):
    matches = []
    for prov in desired_provinces:
        if prov in PROVINCE_URLS:
            matches.append((prov, PROVINCE_URLS[prov]))
    logger.info(f"Using {len(matches)} predefined province URLs.")
    return matches

def extract_city_links(driver, province_name, province_url):
    if not safe_get(driver, province_url):
        return []
    respectful_pause()
    anchors = find_location_anchors(driver)
    results = []
    for txt, href in anchors:
        # Pastikan ini bukan link ke provinsi itu sendiri
        if txt and txt.lower() not in province_name.lower() and len(txt) < 80:
            results.append((txt, href))
    # Hilangkan duplikat
    seen = set()
    uniq = []
    for t, h in results:
        k = (t.lower(), h)
        if k not in seen:
            seen.add(k)
            uniq.append((t, h))
    return uniq

def extract_district_links(driver, province, city, city_url):
    if not safe_get(driver, city_url):
        return []
    respectful_pause()
    anchors = find_location_anchors(driver)
    candidates = []
    for txt, href in anchors:
        if txt and len(txt) <= 80:
            candidates.append((txt, href))
    seen = set()
    uniq = []
    for t, h in candidates:
        k = (t.lower(), h)
        if k not in seen:
            seen.add(k)
            uniq.append((t, h))
    return uniq

# ------------------------------
# CSV WRITER
# ------------------------------
def csv_writer_worker(output_path: str, fieldnames: List[str]):
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        while True:
            row = WRITE_QUEUE.get()
            if row is None:
                break
            writer.writerow(row)
            WRITE_QUEUE.task_done()

# ------------------------------
# DISCOVERY CORE
# ------------------------------
def run_discovery_core(output_path: str, target_provinces: List[str]):
    driver = make_driver(pick_user_agent())
    try:
        provinces = extract_province_links(driver, target_provinces)
        for prov_text, prov_url in provinces:
            prov_name = prov_text
            logger.info(f"Processing province: {prov_name}")
            cities = extract_city_links(driver, prov_name, prov_url)
            if not cities:
                cities = [(prov_name, prov_url)]
            for city_text, city_url in cities:
                districts = extract_district_links(driver, prov_name, city_text, city_url)
                if not districts:
                    WRITE_QUEUE.put({
                        "nation": CONFIG["nation"],
                        "province": prov_name,
                        "city": city_text,
                        "district": city_text,
                        "district_url": city_url,
                        "extracted_at": datetime.utcnow().isoformat()
                    })
                else:
                    for dist_text, dist_url in districts:
                        WRITE_QUEUE.put({
                            "nation": CONFIG["nation"],
                            "province": prov_name,
                            "city": city_text,
                            "district": dist_text,
                            "district_url": dist_url,
                            "extracted_at": datetime.utcnow().isoformat()
                        })
                respectful_pause()
    finally:
        driver.quit()

# ------------------------------
# HELPER FUNCTION
# ------------------------------
def run_location_discovery(province: str, output_csv: str):
    original_targets = CONFIG["target_provinces"].copy()
    matches = [p for p in original_targets if province.lower() in p.lower()]
    if not matches:
        raise ValueError(f"Province '{province}' not found in target_provinces: {original_targets}")

    fieldnames = ["nation", "province", "city", "district", "district_url", "extracted_at"]
    writer_thread = threading.Thread(target=csv_writer_worker, args=(output_csv, fieldnames), daemon=True)
    writer_thread.start()

    start = time.time()
    try:
        run_discovery_core(output_csv, matches)
    finally:
        WRITE_QUEUE.put(None)
        writer_thread.join()
    logger.info(f"✅ Discovery for {matches[0]} done in {time.time()-start:.1f} s. Saved to {output_csv}")

In [2]:
import csv
import logging
import os
import random
import sys
import threading
import time
from datetime import datetime
from queue import Queue
from typing import List, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
from tqdm.notebook import tqdm

# ------------------------------
# KONFIGURASI
# ------------------------------
CONFIG = {
    "input_csv": "city_url_list.csv",
    "output_csv": "district_url_list.csv",
    "max_threads": 2,      # ✅ DIKURANGI menjadi 2 untuk stabilitas maksimum
    "wait_seconds": 15,    # ✅ DINAIKKAN menjadi 15 detik untuk antisipasi halaman berat
    "headless": False, 
    "respectful_delay_min": 1.0,
    "respectful_delay_max": 2.5,
}

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:%(name)s: %(message)s", handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger("olx-district-extractor")
WRITE_QUEUE: "Queue[dict]" = Queue()

def respectful_pause(): time.sleep(random.uniform(CONFIG["respectful_delay_min"], CONFIG["respectful_delay_max"]))

def make_driver():
    driver_path = os.path.join(os.getcwd(), "msedgedriver.exe")
    service = Service(driver_path)
    options = Options()
    if CONFIG["headless"]: options.add_argument("--headless=new")
    options.add_argument("--inprivate")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-geolocation")
    driver = webdriver.Edge(service=service, options=options)
    driver.set_page_load_timeout(CONFIG["wait_seconds"] * 2)
    return driver

def safe_get(driver, url):
    try:
        driver.get(url)
        return True
    except (WebDriverException, TimeoutException) as e:
        logger.warning(f"Gagal memuat {url}: {e}")
        return False

def handle_initial_popups(driver):
    try:
        close_button_xpath = "//button[@aria-label='Close' or @aria-label='close']"
        close_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, close_button_xpath)))
        close_button.click()
        logger.info("Pop-up Ad Blocker ditutup.")
        time.sleep(1)
    except TimeoutException:
        pass

def find_location_anchors(driver) -> List[Tuple[str, str]]:
    anchors = []
    try:
        xpath_selector = "//h3[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'lokasi')]/following-sibling::ul[1]//a"
        WebDriverWait(driver, 5).until(EC.visibility_of_element_located((By.XPATH, xpath_selector)))
        link_elements = driver.find_elements(By.XPATH, xpath_selector)
        for a in link_elements:
            location_name_element = a.find_element(By.XPATH, ".//span[1]")
            txt = (location_name_element.text or "").strip()
            href = a.get_attribute("href")
            if txt and href:
                anchors.append((txt, href.strip()))
    except TimeoutException:
        logger.warning("Tidak menemukan daftar sub-lokasi di sidebar.")
    return anchors

def is_valid_location(text: str, parent_location: str) -> bool:
    text_lower = text.lower()
    if "dijual" in text_lower: return False
    if text_lower in parent_location.lower(): return False
    return True

def csv_writer_worker(output_path: str, fieldnames: List[str]):
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        while True:
            row = WRITE_QUEUE.get()
            if row is None: break
            writer.writerow(row)
            WRITE_QUEUE.task_done()

def extract_districts_from_city(city_data: dict):
    province = city_data.get('province', '')
    city = city_data.get('city', '')
    city_url = city_data.get('city_url', '')
    
    logger.info(f"Memproses kota: {city}, {province}...")
    driver = make_driver()
    try:
        if not safe_get(driver, city_url):
            return
        
        handle_initial_popups(driver)
        respectful_pause()
        
        districts = find_location_anchors(driver)
        
        if not districts:
            logger.warning(f"Tidak ada distrik ditemukan untuk {city}. Menyimpan data level kota.")
            WRITE_QUEUE.put({
                "province": province, "city": city,
                "district": city, "district_url": city_url
            })
        else:
            for dist_text, dist_url in districts:
                if is_valid_location(dist_text, city):
                    WRITE_QUEUE.put({
                        "province": province, "city": city,
                        "district": dist_text, "district_url": dist_url
                    })
    finally:
        driver.quit()

def main():
    try:
        with open(CONFIG["input_csv"], mode='r', encoding='utf-8-sig') as f: # Menggunakan utf-8-sig untuk handle BOM
            cities_to_process = list(csv.DictReader(f))
    except FileNotFoundError:
        logger.error(f"File input tidak ditemukan: {CONFIG['input_csv']}")
        return

    if not cities_to_process:
        logger.warning("Tidak ada kota untuk diproses di file input.")
        return

    fieldnames = ["province", "city", "district", "district_url"]
    writer_thread = threading.Thread(target=csv_writer_worker, args=(CONFIG["output_csv"], fieldnames), daemon=True)
    writer_thread.start()

    start = time.time()
    with ThreadPoolExecutor(max_workers=CONFIG["max_threads"]) as executor:
        futures = [executor.submit(extract_districts_from_city, city_data) for city_data in cities_to_process]
        for future in tqdm(as_completed(futures), total=len(cities_to_process), desc="Mengekstrak Distrik"):
            try:
                future.result()
            except Exception as e:
                logger.error(f"Terjadi error pada salah satu thread: {e}")

    WRITE_QUEUE.put(None)
    writer_thread.join()
    logger.info(f"✅ Selesai! Ekstraksi distrik selesai dalam {time.time()-start:.1f} detik. Hasil disimpan di {CONFIG['output_csv']}")

In [3]:
main()

2025-10-11 18:38:34,139 INFO:olx-district-extractor: Memproses kota: Surabaya Kota, Jawa Timur...
2025-10-11 18:38:34,142 INFO:olx-district-extractor: Memproses kota: Malang Kota, Jawa Timur...


Mengekstrak Distrik:   0%|          | 0/54 [00:00<?, ?it/s]

2025-10-11 18:38:52,478 WARNING:olx-district-extractor: Tidak menemukan daftar sub-lokasi di sidebar.
2025-10-11 18:38:52,478 WARNING:olx-district-extractor: Tidak ada distrik ditemukan untuk Malang Kota. Menyimpan data level kota.
2025-10-11 18:38:52,894 WARNING:olx-district-extractor: Tidak menemukan daftar sub-lokasi di sidebar.
2025-10-11 18:38:52,894 WARNING:olx-district-extractor: Tidak ada distrik ditemukan untuk Surabaya Kota. Menyimpan data level kota.
2025-10-11 18:38:54,760 INFO:olx-district-extractor: Memproses kota: Sidoarjo Kab., Jawa Timur...
2025-10-11 18:38:55,293 INFO:olx-district-extractor: Memproses kota: Malang Kab., Jawa Timur...
2025-10-11 18:39:13,380 WARNING:olx-district-extractor: Tidak menemukan daftar sub-lokasi di sidebar.
2025-10-11 18:39:13,380 WARNING:olx-district-extractor: Tidak ada distrik ditemukan untuk Malang Kab.. Menyimpan data level kota.
2025-10-11 18:39:14,700 WARNING:olx-district-extractor: Tidak menemukan daftar sub-lokasi di sidebar.
2025-1

In [19]:
import os
import time
import csv
import random
import logging
import pandas as pd
import threading
from queue import Queue
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

# ===========================
# KONFIGURASI DASAR
# ===========================
CONFIG = {
    "headless": True,               # ubah ke False jika ingin lihat browser-nya
    "wait_seconds": 20,             # waktu maksimum tunggu elemen
    "retry_limit": 3,               # jumlah percobaan jika gagal load
    "min_delay": 1.2,               # delay acak antar request
    "max_delay": 2.8,
    "max_threads": 3,               # jumlah paralel worker
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s:%(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("olx-city-district-discovery")

# ===========================
# BANTUAN: RANDOM USER AGENT
# ===========================
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edg/120.0 Safari/537.36"
]
def pick_user_agent():
    return random.choice(USER_AGENTS)

# ===========================
# DRIVER SETUP
# ===========================
def make_driver(user_agent=None):
    options = webdriver.EdgeOptions()
    if CONFIG["headless"]:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--log-level=3")
    if user_agent:
        options.add_argument(f"user-agent={user_agent}")

    driver_path = os.path.join(os.getcwd(), "msedgedriver.exe")
    service = EdgeService(executable_path=driver_path)
    driver = webdriver.Edge(service=service, options=options)
    driver.set_page_load_timeout(CONFIG["wait_seconds"] * 2)
    return driver

# ===========================
# PROSES EKSTRAKSI DISTRIK
# ===========================
def extract_district_links(driver, city_name, city_url, province):
    for attempt in range(CONFIG["retry_limit"]):
        try:
            logger.info(f"🔍 [{province}] {city_name} | attempt {attempt+1} → {city_url}")
            driver.get(city_url)
            time.sleep(random.uniform(CONFIG["min_delay"], CONFIG["max_delay"]) + 2)

            sidebar = WebDriverWait(driver, CONFIG["wait_seconds"]).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "aside"))
            )

            anchors = sidebar.find_elements(By.TAG_NAME, "a")
            districts = []
            for a in anchors:
                link = a.get_attribute("href")
                text = a.text.strip()
                if link and "/dijual-rumah-apartemen/" in link and text:
                    districts.append({
                        "province": province,
                        "city": city_name,
                        "district": text,
                        "url": link
                    })

            if districts:
                logger.info(f"✅ Found {len(districts)} district links for {city_name}")
                return districts
            else:
                logger.warning(f"⚠️ No district links found for {city_name}, retrying...")
                time.sleep(3)
        except TimeoutException:
            logger.warning(f"⏰ Timeout loading page for {city_name}, retrying...")
        except WebDriverException as e:
            logger.warning(f"⚠️ WebDriver error on {city_name}: {e}")
            time.sleep(4)
    logger.error(f"❌ Failed to extract districts for {city_name} after {CONFIG['retry_limit']} attempts.")
    return []

# ===========================
# WORKER THREAD FUNCTION
# ===========================
def worker(input_queue, output_csv, lock):
    driver = make_driver(pick_user_agent())
    while not input_queue.empty():
        try:
            province, city, city_url = input_queue.get_nowait()
        except:
            break
        try:
            districts = extract_district_links(driver, city, city_url, province)
            if districts:
                with lock:
                    df_temp = pd.DataFrame(districts)
                    df_temp.to_csv(output_csv, mode="a", header=not os.path.exists(output_csv), index=False)
        finally:
            input_queue.task_done()
    driver.quit()

# ===========================
# MAIN DISCOVERY FUNCTION
# ===========================
def discover_districts_from_city_list(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    input_queue = Queue()
    lock = threading.Lock()

    for _, row in df.iterrows():
        input_queue.put((row["province"], row["city_name"], row["city_url"]))

    threads = []
    start = time.time()

    for i in range(CONFIG["max_threads"]):
        t = threading.Thread(target=worker, args=(input_queue, output_csv, lock))
        t.daemon = True
        threads.append(t)
        t.start()

    input_queue.join()
    for t in threads:
        t.join()

    duration = time.time() - start
    logger.info(f"✅ Discovery done in {duration:.1f} s. Results saved to {output_csv}")

In [20]:
discover_districts_from_city_list("city_url_list.csv", "districts_jatim.csv")

Exception ignored in: <function tqdm.__del__ at 0x0000021EB3627420>
Traceback (most recent call last):
  File "c:\Users\Ryan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\Ryan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


KeyError: 'city_name'

## Tes Provinsi Jawa Timur

In [5]:
run_location_discovery(province="Jawa Timur", output_csv="districts_jatim.csv")

2025-10-11 19:04:58,523 INFO:olx-location-discovery: Using 1 predefined province URLs.
2025-10-11 19:04:58,523 INFO:olx-location-discovery: Processing province: Jawa Timur
2025-10-11 19:05:16,348 INFO:olx-location-discovery: ✅ Discovery for Jawa Timur done in 19.0 s. Saved to districts_jatim.csv


In [19]:
import time
import csv
import random
import logging
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException

# ==================== KONFIGURASI ====================
BASE_URL = "https://www.olx.co.id/properti/rumah/"
OUTPUT_CSV = "province_url_list.csv"
WAIT_TIME = 15
PAUSE = (1.5, 2.5)
MAX_RETRY = 3

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO
)
logger = logging.getLogger("olx-location-extractor")


# ==================== DRIVER SETUP ====================
def make_driver():
    driver_path = os.path.join(os.getcwd(), "msedgedriver.exe")
    if not os.path.exists(driver_path):
        raise FileNotFoundError("❌ msedgedriver.exe tidak ditemukan di folder kerja.")
    
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    driver = webdriver.Edge(service=Service(driver_path), options=opts)
    logger.info(f"✅ Driver ditemukan: {driver_path}")
    return driver


# ==================== HELPER ====================
def safe_click_js(driver, element):
    driver.execute_script("arguments[0].scrollIntoView(true);", element)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", element)
    time.sleep(1.0)


def wait_for(driver, css, timeout=WAIT_TIME):
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, css))
    )


def save_snapshot(driver, name):
    with open(f"{name}.html", "w", encoding="utf-8") as f:
        f.write(driver.page_source)
    logger.warning(f"📸 Snapshot halaman disimpan ke {name}.html")


# ==================== MAIN SCRAPER ====================
def extract_all_provinces():
    driver = make_driver()
    results = []

    for attempt in range(1, MAX_RETRY + 1):
        try:
            logger.info(f"🌐 Membuka halaman: {BASE_URL}")
            driver.get(BASE_URL)

            wait_for(driver, "body")
            logger.info("✅ Halaman kategori berhasil dimuat.")
            time.sleep(random.uniform(*PAUSE))

            # Tombol lokasi (struktur baru OLX)
            try:
                location_button = WebDriverWait(driver, WAIT_TIME).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'button[data-aut-id*="collapsible_topLocations"]'))
                )
                logger.info("📍 Tombol 'Lokasi' berhasil ditemukan.")
            except TimeoutException:
                raise RuntimeError("❌ Tidak menemukan tombol 'Lokasi' di halaman (struktur baru).")

            # Klik tombol lokasi
            try:
                safe_click_js(driver, location_button)
                logger.info("✅ Klik tombol 'Lokasi' berhasil.")
            except ElementClickInterceptedException:
                logger.warning("⚠️ Klik biasa gagal, coba pakai JS.")
                safe_click_js(driver, location_button)

            # Tunggu daftar lokasi muncul
            time.sleep(3)
            save_snapshot(driver, "after_click_location")

            # Cari daftar provinsi
            province_links = driver.find_elements(By.CSS_SELECTOR, 'ul[data-aut-id="collapsible_topLocations_list"] li a')
            if not province_links:
                province_links = driver.find_elements(By.CSS_SELECTOR, 'ul._2lVe2 li a')

            if not province_links:
                raise RuntimeError("❌ Tidak menemukan daftar provinsi setelah membuka 'Lokasi'.")

            logger.info(f"🔍 Ditemukan {len(province_links)} provinsi.")

            for a in province_links:
                name = a.text.strip()
                href = a.get_attribute("href")
                if name and href:
                    results.append([name, href])

            break  # sukses keluar dari retry loop

        except Exception as e:
            logger.error(f"⚠️ Percobaan {attempt}/{MAX_RETRY} gagal: {e}")
            save_snapshot(driver, f"failed_page_attempt_{attempt}")
            if attempt < MAX_RETRY:
                logger.info("🔁 Mencoba ulang dalam 5 detik...")
                time.sleep(5)
            else:
                logger.critical("🚨 Semua percobaan gagal.")
                raise e

    driver.quit()
    logger.info("🧹 Browser ditutup.")

    # Simpan hasil
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["province_name", "province_url"])
        writer.writerows(results)

    logger.info(f"🎯 Selesai! Total provinsi ditemukan: {len(results)}. Hasil disimpan ke {OUTPUT_CSV}")
    return results


# ==================== JALANKAN ====================
results = extract_all_provinces()
results[:10]

2025-10-11 20:42:02,951 INFO:olx-location-extractor: ✅ Driver ditemukan: c:\Users\Ryan\Documents\tugas_akhir\tugas_akhir\Coding\Data Gathering\Link Extraction\msedgedriver.exe
2025-10-11 20:42:02,952 INFO:olx-location-extractor: 🌐 Membuka halaman: https://www.olx.co.id/properti/rumah/
2025-10-11 20:42:03,357 INFO:olx-location-extractor: ✅ Halaman kategori berhasil dimuat.
2025-10-11 20:42:20,697 ERROR:olx-location-extractor: ⚠️ Percobaan 1/3 gagal: ❌ Tidak menemukan tombol 'Lokasi' di halaman (struktur baru).
2025-10-11 20:42:20,715 WARNING:olx-location-extractor: 📸 Snapshot halaman disimpan ke failed_page_attempt_1.html
2025-10-11 20:42:20,715 INFO:olx-location-extractor: 🔁 Mencoba ulang dalam 5 detik...
2025-10-11 20:42:25,717 INFO:olx-location-extractor: 🌐 Membuka halaman: https://www.olx.co.id/properti/rumah/
2025-10-11 20:42:25,799 INFO:olx-location-extractor: ✅ Halaman kategori berhasil dimuat.
2025-10-11 20:42:43,238 ERROR:olx-location-extractor: ⚠️ Percobaan 2/3 gagal: ❌ Tidak 

RuntimeError: ❌ Tidak menemukan tombol 'Lokasi' di halaman (struktur baru).

## Cek Hasil CSV Jawa Timur

In [ ]:
import pandas as pd

df = pd.read_csv("districts_jatim.csv")
df.head(10)

## Cek Outlier

In [5]:
import pandas as pd

df = pd.read_csv("districts_jatim.csv")
outliers = df[~df['district_url'].str.contains("/dijual-rumah-apartemen_c5158")]
print(outliers)

Empty DataFrame
Columns: [nation, province, city, district, district_url, extracted_at]
Index: []


## Konversi ke Excel

In [6]:
import pandas as pd

input_csv = "districts_jatim.csv"
output_excel = "districts_jatim.xlsx"

try:
    df = pd.read_csv(input_csv)

    df.to_excel(output_excel, index=False)

    print(f"✅ File berhasil dikonversi ke: {output_excel}")

except Exception as e:
    print(f"❌ Terjadi error: {e}")

✅ File berhasil dikonversi ke: districts_jatim.xlsx
